In [1]:
import numpy as np
import random

class GridWorldMDP:
    def __init__(self, size, goal, trap):
        self.size = size
        self.goal = goal
        self.trap = trap

        self.state_space = [(i, j) for i in range(size) for j in range(size)]
        self.action_space = ["UP", "DOWN", "LEFT", "RIGHT"]

        self.transitions = self.build_transitions()
        self.rewards = self.build_rewards()

    def build_transitions(self):
        transitions = {}
        for state in self.state_space:
            transitions[state] = {}
            for action in self.action_space:
                transitions[state][action] = self.calculate_transition(state, action)
        return transitions

    def calculate_transition(self, state, action):
        i, j = state

        if action == "UP":
            next_state = (i - 1, j)
        elif action == "DOWN":
            next_state = (i + 1, j)
        elif action == "LEFT":
            next_state = (i, j - 1)
        elif action == "RIGHT":
            next_state = (i, j + 1)

        # Boundary check
        next_state = (
            max(0, min(next_state[0], self.size - 1)),
            max(0, min(next_state[1], self.size - 1))
        )

        # Terminal states stay same
        if state == self.goal or state == self.trap:
            return [(1.0, state)]

        return [(1.0, next_state)]

    def build_rewards(self):
        rewards = {}
        for state in self.state_space:
            rewards[state] = -1.0  # default

        rewards[self.goal] = 0.0
        rewards[self.trap] = -10.0

        return rewards


# ---------------- VALUE ITERATION ----------------
def value_iteration(mdp, gamma=0.9, epsilon=0.01):
    V = {state: 0 for state in mdp.state_space}

    while True:
        delta = 0

        for state in mdp.state_space:
            if state == mdp.goal or state == mdp.trap:
                continue

            v = V[state]

            V[state] = max(
                sum(p * (mdp.rewards[next_state] + gamma * V[next_state])
                    for p, next_state in mdp.transitions[state][action])
                for action in mdp.action_space
            )

            delta = max(delta, abs(v - V[state]))

        if delta < epsilon:
            break

    return V


# ---------------- POLICY ITERATION ----------------
def policy_iteration(mdp, gamma=0.9):
    policy = {
        state: random.choice(mdp.action_space)
        for state in mdp.state_space
        if state != mdp.goal and state != mdp.trap
    }

    V = {state: 0 for state in mdp.state_space}

    while True:
        # Policy Evaluation
        while True:
            delta = 0
            for state in mdp.state_space:
                if state == mdp.goal or state == mdp.trap:
                    continue

                v = V[state]
                action = policy[state]

                V[state] = sum(
                    p * (mdp.rewards[next_state] + gamma * V[next_state])
                    for p, next_state in mdp.transitions[state][action]
                )

                delta = max(delta, abs(v - V[state]))

            if delta < 0.01:
                break

        # Policy Improvement
        policy_stable = True

        for state in mdp.state_space:
            if state == mdp.goal or state == mdp.trap:
                continue

            old_action = policy[state]

            policy[state] = max(
                mdp.action_space,
                key=lambda action: sum(
                    p * (mdp.rewards[next_state] + gamma * V[next_state])
                    for p, next_state in mdp.transitions[state][action]
                )
            )

            if old_action != policy[state]:
                policy_stable = False

        if policy_stable:
            break

    return policy, V


# ---------------- MAIN ----------------
if __name__ == "__main__":
    size = 3
    goal = (2, 2)
    trap = (1, 1)

    mdp = GridWorldMDP(size, goal, trap)

    print("=== Value Iteration ===")
    V = value_iteration(mdp)
    for state in V:
        print(f"{state} : {V[state]:.2f}")

    print("\n=== Policy Iteration ===")
    policy, V = policy_iteration(mdp)
    for state in policy:
        print(f"{state} -> {policy[state]}, Value: {V[state]:.2f}")

=== Value Iteration ===
(0, 0) : -2.71
(0, 1) : -1.90
(0, 2) : -1.00
(1, 0) : -1.90
(1, 1) : 0.00
(1, 2) : 0.00
(2, 0) : -1.00
(2, 1) : 0.00
(2, 2) : 0.00

=== Policy Iteration ===
(0, 0) -> DOWN, Value: -2.71
(0, 1) -> RIGHT, Value: -1.90
(0, 2) -> DOWN, Value: -1.00
(1, 0) -> DOWN, Value: -1.90
(1, 2) -> DOWN, Value: 0.00
(2, 0) -> RIGHT, Value: -1.00
(2, 1) -> RIGHT, Value: 0.00
